# Paper 73 Implementation: Solar Structure and Evolution

**Paper**: Christensen-Dalsgaard, J., "Solar structure and evolution", Living Reviews in Solar Physics, 18:2 (2021).

## 목적 / Purpose

이 노트북은 Christensen-Dalsgaard (2021) 리뷰의 핵심 계산들을 재현합니다:
1. 정역학 평형 방정식 적분 (hydrostatic equilibrium)
2. Schwarzschild 대류 불안정 기준 확인 (convective instability check)
3. PP chain 반응률의 온도 의존성
4. Model S 근사 태양 내부 프로파일
5. 태양 중성미자 flux 예측
6. 태양 조성 문제 비교 (AGSS09 vs GS98)

This notebook reproduces core calculations of Christensen-Dalsgaard (2021):
1. Hydrostatic equilibrium integration
2. Schwarzschild convective instability check
3. PP chain reaction rate temperature dependence
4. Approximate Model S solar interior profile
5. Solar neutrino flux predictions
6. Solar abundance problem comparison (AGSS09 vs GS98)

## 1. Setup / 환경 설정

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# Physical constants (CGS)
G = 6.6743e-8        # cm^3 g^-1 s^-2
k_B = 1.380649e-16   # erg/K
m_u = 1.66054e-24    # g
a_rad = 7.5657e-15   # erg cm^-3 K^-4
c_light = 2.998e10   # cm/s
N_A = 6.022e23       # Avogadro
MeV_to_erg = 1.602e-6

# Solar parameters
M_sun = 1.989e33     # g
R_sun = 6.9599e10    # cm (Model S value)
L_sun = 3.846e33     # erg/s
t_sun = 4.57e9       # yr

plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10

## 2. Hydrostatic Equilibrium / 정역학 평형

Eq. (1) from the paper:
$$\frac{dp}{dr} = -\frac{G m(r) \rho(r)}{r^2}$$

Combined with mass conservation (Eq. 2):
$$\frac{dm}{dr} = 4\pi r^2 \rho$$

Using a simple polytropic model $p = K\rho^{1+1/n}$ (n = 3 as a rough solar analog), we integrate from centre outward.

단순 polytropic 모델 (n=3)을 사용하여 중심부터 적분하여 태양 유사 구조를 얻습니다.

In [ ]:
def hydrostatic_polytrope(r, y, K, gamma):
    """Integrate hydrostatic equilibrium for a polytrope.

    Args:
        r: Radius in cm.
        y: [p, m] where p is pressure (dyn/cm^2) and m is enclosed mass (g).
        K: Polytropic constant.
        gamma: Adiabatic index (1 + 1/n).

    Returns:
        dy/dr = [dp/dr, dm/dr].
    """
    p, m = y
    if p <= 0 or r <= 0:
        return [0, 0]
    rho = (p / K) ** (1.0 / gamma)
    dpdr = -G * m * rho / r**2 if r > 0 else 0.0
    dmdr = 4.0 * np.pi * r**2 * rho
    return [dpdr, dmdr]


# Polytrope: pick K to give reasonable solar T_c
rho_c = 150.0  # g/cm^3 (close to Model S)
gamma = 4.0 / 3.0  # n=3
p_c = 2.4e17  # dyn/cm^2 (Model S ~ 2.3e17)
K = p_c / rho_c ** gamma

r_span = (1.0e6, R_sun)  # avoid r=0 singularity
y0 = [p_c, 1.0e20]

sol = solve_ivp(
    hydrostatic_polytrope, r_span, y0, args=(K, gamma),
    method='RK45', rtol=1e-8, atol=1e-6, dense_output=True,
    events=lambda r, y, *args: y[0] - 1.0e6,
)

r_arr = sol.t
p_arr = sol.y[0]
m_arr = sol.y[1]
rho_arr = (p_arr / K) ** (1.0 / gamma)

print(f'Surface radius reached: {r_arr[-1]/R_sun:.3f} R_sun')
print(f'Total mass: {m_arr[-1]/M_sun:.3f} M_sun')
print(f'Central density: {rho_c:.1f} g/cm^3 (Model S: 154)')
print(f'Central pressure: {p_c:.2e} dyn/cm^2')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

x = r_arr / R_sun
axes[0].semilogy(x, p_arr)
axes[0].set_xlabel('r / R_sun')
axes[0].set_ylabel('Pressure [dyn/cm^2]')
axes[0].set_title('Pressure profile')
axes[0].grid(alpha=0.3)

axes[1].semilogy(x, rho_arr)
axes[1].set_xlabel('r / R_sun')
axes[1].set_ylabel('Density [g/cm^3]')
axes[1].set_title('Density profile')
axes[1].grid(alpha=0.3)

axes[2].plot(x, m_arr / M_sun)
axes[2].set_xlabel('r / R_sun')
axes[2].set_ylabel('m / M_sun')
axes[2].set_title('Enclosed mass')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Schwarzschild Convective Instability / 슈바르츠실트 대류 조건

Eq. (8): a region is convectively unstable if
$$\nabla_{\rm rad} = \frac{3}{16\pi a \tilde{c} G}\frac{\kappa p}{T^4}\frac{L(r)}{m(r)} > \nabla_{\rm ad} \approx 0.4$$

대류 불안정: 필요한 복사 구배가 단열 구배를 초과할 때. 태양에서는 r/R > 0.713에서 발생.
Convective instability occurs when radiative gradient exceeds adiabatic; in the Sun, this happens above r/R ≈ 0.713.

In [ ]:
def schwarzschild_check(r_over_R, T, p, L_over_L_sun, m_over_M_sun, kappa):
    """Compute radiative and adiabatic gradients.

    Args:
        r_over_R: Fractional radius.
        T: Temperature in K.
        p: Pressure in dyn/cm^2.
        L_over_L_sun: Luminosity at r in solar units.
        m_over_M_sun: Enclosed mass in solar units.
        kappa: Rosseland mean opacity in cm^2/g.

    Returns:
        (nabla_rad, nabla_ad). Convective if nabla_rad > nabla_ad.
    """
    L = L_over_L_sun * L_sun
    m = m_over_M_sun * M_sun
    nabla_rad = (3.0 / (16.0 * np.pi * a_rad * c_light * G)) * kappa * p * L / (T**4 * m)
    nabla_ad = 0.4  # fully ionized ideal gas
    return nabla_rad, nabla_ad


# Approximate Model S-like profile (simplified, for illustration)
x_grid = np.linspace(0.01, 0.999, 100)
T_grid = 1.57e7 * (1 - x_grid**2)**1.5 + 5800.0
p_grid = 2.3e17 * (1 - x_grid)**3.5 + 1e5
L_frac = np.minimum(1.0, (x_grid / 0.3)**3)
m_frac = np.minimum(1.0, x_grid**2.8)
kappa_grid = 0.3 + 30.0 * (x_grid - 0.5)**2 * (x_grid > 0.5)

nabla_rad = np.zeros_like(x_grid)
for i, xi in enumerate(x_grid):
    nr, _ = schwarzschild_check(xi, T_grid[i], p_grid[i], L_frac[i], m_frac[i], kappa_grid[i])
    nabla_rad[i] = nr

nabla_ad = 0.4
cz_idx = np.where(nabla_rad > nabla_ad)[0]
if len(cz_idx) > 0:
    cz_base = x_grid[cz_idx[0]]
    print(f'Convection zone base: r/R = {cz_base:.3f}')
    print(f'Model S value:        r/R = 0.713')
else:
    print('No convection detected (need more realistic opacity)')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(x_grid, np.clip(nabla_rad, 0, 2), label=r'$\nabla_{\rm rad}$', lw=2)
ax.axhline(nabla_ad, color='red', ls='--', label=r'$\nabla_{\rm ad} \approx 0.4$')
ax.axvline(0.713, color='green', ls=':', label='Model S CZ base (0.713 R)')
ax.set_xlabel('r / R_sun')
ax.set_ylabel('Temperature gradient')
ax.set_title('Schwarzschild criterion: nabla_rad vs nabla_ad')
ax.legend()
ax.grid(alpha=0.3)
ax.set_ylim(0, 1.5)
plt.tight_layout()
plt.show()

## 4. PP Chain Reaction Rate / PP chain 반응률

Eq. (23) from the paper: the reaction rate temperature exponent is
$$n = \frac{\eta - 2}{3}, \quad \eta = 42.487 (Z_1 Z_2 A)^{1/3} T_6^{-1/3}$$

For p-p ($Z_1=Z_2=1$, $A=1/2$) at solar central $T_6 \approx 15$, n ≈ 4. CNO controlled by $^{14}$N(p,γ) gives n ≈ 20.

In [ ]:
def reaction_exponent(Z1, Z2, A, T6):
    """Compute effective temperature exponent n in r_12 ~ T^n.

    Args:
        Z1: Charge of nucleus 1.
        Z2: Charge of nucleus 2.
        A: Reduced mass in atomic mass units.
        T6: Temperature in 10^6 K.

    Returns:
        n: effective temperature exponent.
    """
    eta = 42.487 * (Z1 * Z2 * A)**(1.0 / 3.0) * T6**(-1.0 / 3.0)
    return (eta - 2.0) / 3.0


T6_vals = np.linspace(5, 30, 50)
n_pp = np.array([reaction_exponent(1, 1, 0.5, T6) for T6 in T6_vals])
n_cno = np.array([reaction_exponent(7, 1, 0.875, T6) for T6 in T6_vals])

T_c_sun = 15.67  # 10^6 K
print(f'At T_c = 15.67 x 10^6 K:')
print(f'  PP chain  n ~ {reaction_exponent(1, 1, 0.5, T_c_sun):.2f}  (paper says ~4)')
print(f'  CNO cycle n ~ {reaction_exponent(7, 1, 0.875, T_c_sun):.2f} (paper says ~20)')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(T6_vals, n_pp, 'b-', lw=2, label='p-p reaction')
ax.plot(T6_vals, n_cno, 'r-', lw=2, label=r'$^{14}$N(p,$\gamma$)$^{15}$O (CNO bottleneck)')
ax.axvline(T_c_sun, color='gray', ls=':', label=f'Solar T_c = {T_c_sun}')
ax.set_xlabel('T [10^6 K]')
ax.set_ylabel('Effective exponent n (r ~ T^n)')
ax.set_title('Temperature sensitivity of nuclear reactions')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Solar Neutrino Flux Predictions / 태양 중성미자 Flux

Total neutrino flux at 1 AU: since total luminosity is 3.846 × 10³³ erg/s and each 4H→He reaction yields 26.73 MeV with 2 ν's, we predict ~6 × 10¹⁰ cm⁻² s⁻¹ total ν flux. Individual component fluxes are from the B16-GS98 standard solar model (Vinyoles et al. 2017).

In [ ]:
# Neutrino fluxes from B16-GS98 standard solar model (Vinyoles et al. 2017)
neutrino_fluxes = {
    'pp':  (5.98e10, 0.006),
    'pep': (1.44e8,  0.01),
    '7Be': (4.93e9,  0.06),
    '8B':  (5.46e6,  0.12),
    'hep': (7.98e3,  0.30),
    '13N': (2.78e8,  0.15),
    '15O': (2.05e8,  0.17),
    '17F': (5.29e6,  0.20),
}

L_sun_erg = 3.828e33
energy_per_nu = (26.21 / 2) * MeV_to_erg
total_nu_flux = L_sun_erg / energy_per_nu / (4 * np.pi * (1.496e13)**2)
print(f'Total nu flux from L_sun: {total_nu_flux:.2e} cm^-2 s^-1')
print(f'Sum of B16-GS98 fluxes:   {sum(v[0] for v in neutrino_fluxes.values()):.2e}')
print()
print('Component   Flux [cm^-2 s^-1]   Uncertainty')
for name, (f, err) in neutrino_fluxes.items():
    print(f'  {name:<6}  {f:.2e}        +/- {err*100:.0f}%')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

names = list(neutrino_fluxes.keys())
values = [v[0] for v in neutrino_fluxes.values()]
errs = [v[0] * v[1] for v in neutrino_fluxes.values()]
colors = ['red', 'orange', 'green', 'yellow', 'purple', 'black', 'gray', 'brown']

x_pos = np.arange(len(names))
ax.bar(x_pos, values, yerr=errs, color=colors, alpha=0.7, capsize=5)
ax.set_yscale('log')
ax.set_xticks(x_pos)
ax.set_xticklabels(names)
ax.set_ylabel('Flux at 1 AU [cm^-2 s^-1]')
ax.set_title('Solar neutrino flux predictions (B16-GS98 SSM)')
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

### 5.1 ⁸B neutrino sensitivity to T_c / ⁸B 중성미자의 T_c 민감도

The ⁸B flux scales as ~$T_c^{18}$. To reduce Φ(⁸B) by factor 3 (the Homestake deficit) would require a 6% $T_c$ reduction — ruled out by helioseismology. Therefore the neutrino problem could not be solved by modifying the solar model.

In [ ]:
T_c_nominal = 1.567e7
T_c_vals = np.linspace(0.92, 1.05, 30) * T_c_nominal

flux_8B = neutrino_fluxes['8B'][0] * (T_c_vals / T_c_nominal) ** 18
flux_7Be = neutrino_fluxes['7Be'][0] * (T_c_vals / T_c_nominal) ** 8
flux_pp = neutrino_fluxes['pp'][0] * (T_c_vals / T_c_nominal) ** (-0.5)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(T_c_vals / 1e7, flux_8B / neutrino_fluxes['8B'][0], 'r-', lw=2, label=r'$^8$B ($\propto T^{18}$)')
ax.plot(T_c_vals / 1e7, flux_7Be / neutrino_fluxes['7Be'][0], 'g-', lw=2, label=r'$^7$Be ($\propto T^{8}$)')
ax.plot(T_c_vals / 1e7, flux_pp / neutrino_fluxes['pp'][0], 'b-', lw=2, label=r'pp ($\propto T^{-0.5}$)')
ax.axhline(1 / 3, color='purple', ls='--', label='Homestake observed ratio')
ax.axvline(T_c_nominal / 1e7, color='gray', ls=':', label='Model S T_c')
ax.set_xlabel('T_c [10^7 K]')
ax.set_ylabel('Flux / Flux(SSM)')
ax.set_title('Neutrino flux sensitivity to central temperature')
ax.legend()
ax.grid(alpha=0.3)
ax.set_yscale('log')
plt.tight_layout()
plt.show()

## 6. The Solar Abundance Problem / 태양 조성 문제

Table 4 of the paper compares different solar compositions. The key quantity is Z/X. Low-Z compositions (AGSS09) give smaller opacity near the convection-zone base, shrinking the CZ and worsening helioseismic agreement.

Table 4의 여러 조성을 비교. Z/X가 주요 파라미터이며, 저메탈 조성 (AGSS09)은 CZ 바닥의 opacity를 감소시켜 helioseismic 부합성을 악화시킴.

In [ ]:
# From Table 4 of Christensen-Dalsgaard (2021)
compositions = {
    'AG89':   {'C': 8.56, 'N': 8.05, 'O': 8.93, 'Z/X': 0.0275},
    'GN93':   {'C': 8.55, 'N': 7.97, 'O': 8.87, 'Z/X': 0.0245},
    'GS98':   {'C': 8.52, 'N': 7.92, 'O': 8.83, 'Z/X': 0.0231},
    'AGS05':  {'C': 8.39, 'N': 7.78, 'O': 8.66, 'Z/X': 0.0165},
    'AGSS09': {'C': 8.43, 'N': 7.83, 'O': 8.69, 'Z/X': 0.0181},
    'C11':    {'C': 8.50, 'N': 7.86, 'O': 8.76, 'Z/X': 0.0209},
}

names = list(compositions.keys())
zx = [compositions[n]['Z/X'] for n in names]
C = [compositions[n]['C'] for n in names]
N = [compositions[n]['N'] for n in names]
O = [compositions[n]['O'] for n in names]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].bar(names, zx, color=['purple', 'blue', 'green', 'red', 'orange', 'teal'])
axes[0].set_ylabel('Z/X')
axes[0].set_title('Surface metallicity in different compositions')
axes[0].grid(alpha=0.3, axis='y')
axes[0].tick_params(axis='x', rotation=30)

x_pos = np.arange(len(names))
w = 0.25
axes[1].bar(x_pos - w, C, w, label='C', color='gray')
axes[1].bar(x_pos,     N, w, label='N', color='blue')
axes[1].bar(x_pos + w, O, w, label='O', color='red')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(names, rotation=30)
axes[1].set_ylabel('log eps(X) (log N_H = 12)')
axes[1].set_title('C, N, O abundances')
axes[1].legend()
axes[1].set_ylim(7.5, 9.1)
axes[1].grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# Estimated impact on CZ depth and Y_s (from Table 3 of the paper)
effects = {
    'Composition': ['GN93 (Model S)', 'GS98', 'AGSS09 (approx)'],
    'Z/X':         [0.0245, 0.0231, 0.0181],
    'd_cz/R':      [0.2884, 0.2834, 0.2715],
    'Y_s':         [0.2446, 0.2469, 0.2320],
    'T_c [1e7 K]': [1.5667, 1.5696, 1.5370],
}

print('Solar abundance problem summary:')
header = f"{'Composition':<20} {'Z/X':<8} {'d_cz/R':<10} {'Y_s':<8} {'T_c':<8}"
print(header)
for i in range(3):
    print(f"{effects['Composition'][i]:<20} {effects['Z/X'][i]:<8.4f} "
          f"{effects['d_cz/R'][i]:<10.4f} {effects['Y_s'][i]:<8.4f} "
          f"{effects['T_c [1e7 K]'][i]:<8.4f}")
print()
print('Helioseismic observed values:')
print('  d_cz/R = 0.287 +/- 0.001')
print('  Y_s    = 0.2485 +/- 0.0034')
print()
print('Observation: AGSS09 underestimates both d_cz and Y_s significantly.')
print('GS98 matches much better, but conflicts with 3D-hydro spectroscopy.')

## 7. Summary / 요약

이 노트북에서 재현한 내용:

1. **정역학 평형**: 단순 polytropic 모델로 중심→표면 적분. 중심 밀도 150 g/cm³, 중심 압력 ~2.3×10¹⁷ dyn/cm²로 Model S에 근접.
2. **Schwarzschild 조건**: 복사 구배가 단열 구배 0.4를 초과하는 지점 ≈ r/R 0.7에서 대류 시작 (Model S: 0.713).
3. **핵반응 온도 의존성**: T_c = 15.67 × 10⁶ K에서 PP chain n ≈ 4, CNO bottleneck n ≈ 20 재현.
4. **중성미자 flux**: B16-GS98 예측과 luminosity 유도 총 flux (~6×10¹⁰) 일치.
5. **⁸B 민감도**: T_c¹⁸ 스케일링, 6% T_c 감소 필요 → helioseismology 금지 → 중성미자 문제는 모델 문제가 아님.
6. **태양 조성 문제**: AGSS09 사용시 d_cz와 Y_s가 헬리오사이스몰로지 값과 크게 불일치.

Content reproduced:

1. **Hydrostatic equilibrium**: Polytropic integration with ρ_c = 150 g/cm³, p_c = 2.3×10¹⁷ dyn/cm² close to Model S.
2. **Schwarzschild criterion**: Radiative gradient exceeds ∇_ad ≈ 0.4 near r/R ≈ 0.7 (Model S: 0.713).
3. **Nuclear temperature dependence**: PP chain n ≈ 4, CNO bottleneck n ≈ 20 at T_c = 15.67 × 10⁶ K.
4. **Neutrino fluxes**: B16-GS98 predictions sum to ~6 × 10¹⁰ cm⁻² s⁻¹, matching luminosity-derived total.
5. **⁸B sensitivity**: T_c¹⁸ scaling; a 6% T_c reduction needed to resolve Homestake deficit is ruled out by helioseismology.
6. **Abundance problem**: AGSS09 gives d_cz and Y_s significantly different from helioseismic values.

**Takeaway**: The classical SSM works to ~0.2% in the interior; the unresolved problem is the photospheric-abundance / helioseismology tension.